# The CROP algorithm from first principles

CROP turns one error-risk score per step into a certified prefix. This notebook implements the complete algorithm with NumPy and synthetic traces; it requires no dataset, model, or GPU.

## 1. Create calibration and test traces

Each trace has binary step-error labels and noisy verifier scores. Larger scores mean greater error risk. We split by complete trace.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)

def sample_trace():
    n_steps = int(rng.integers(4, 11))
    labels = np.zeros(n_steps, dtype=int)
    if rng.random() < 0.45:
        labels[int(rng.integers(1, n_steps))] = 1
    scores = np.clip(0.12 + 0.72 * labels + rng.normal(0, 0.16, n_steps), 0, 1)
    return labels, scores

examples = [sample_trace() for _ in range(800)]
calibration, test = examples[:500], examples[500:]

## 2. Implement CROP

For a threshold $\lambda$, retain steps until the first score above $\lambda$. A calibration trace incurs loss 1 if its retained prefix contains an error. CROP chooses the largest threshold whose plus-one corrected average loss is at most $\alpha$.

In [ ]:
def prefix_length(scores, threshold):
    rejected = np.flatnonzero(scores > threshold)
    return int(rejected[0]) if len(rejected) else len(scores)

def contamination_loss(labels, scores, threshold):
    length = prefix_length(scores, threshold)
    return int(np.any(labels[:length]))

def fit_crop(calibration_data, alpha=0.05):
    candidates = np.r_[-np.inf, np.unique(np.concatenate([s for _, s in calibration_data]))]
    corrected_risks = []
    for threshold in candidates:
        losses = [contamination_loss(y, s, threshold) for y, s in calibration_data]
        corrected_risks.append((1 + sum(losses)) / (len(losses) + 1))
    corrected_risks = np.asarray(corrected_risks)
    feasible = np.flatnonzero(corrected_risks <= alpha)
    index = int(feasible[-1]) if len(feasible) else 0
    return candidates[index], corrected_risks[index]


## 3. Calibrate once, then certify new traces

Labels are used only during calibration. Deployment needs only the ordered step scores.

In [ ]:
alpha = 0.05
threshold, calibration_risk = fit_crop(calibration, alpha)

retained = np.asarray([prefix_length(scores, threshold) for _, scores in test])
test_losses = np.asarray([contamination_loss(y, s, threshold) for y, s in test])
retained_fraction = np.mean([m / len(y) for m, (y, _) in zip(retained, test)])

print(f'threshold: {threshold:.3f}')
print(f'corrected calibration risk: {calibration_risk:.3f}')
print(f'test prefix-contamination risk: {test_losses.mean():.3f}')
print(f'mean prefix retained: {retained_fraction:.3f}')

## 4. Use the packaged API

The public API performs the same calculation and validates score/trace alignment.

In [ ]:
from types import SimpleNamespace
from crop import fit_prefix_calibrator

cal_traces = [SimpleNamespace(steps=[None] * len(y), y_errors=y) for y, _ in calibration]
cal_scores = [scores for _, scores in calibration]
test_scores = [scores for _, scores in test]

fitted = fit_prefix_calibrator(cal_traces, cal_scores, alpha=alpha, grid_size=10_000)
assert fitted.threshold == threshold
assert np.array_equal(fitted.predict(test_scores), retained)
fitted

That is the complete CROP layer: define a prefix-contamination loss, calibrate a threshold on exchangeable complete traces, and apply the threshold to new step scores. The verifier that produces those scores can be replaced without changing CROP.